# Qwen 4B 风险识别 — 基线测试 + LoRA 微调

### 使用步骤
1. 菜单栏 → 运行时 → 更改运行时类型 → 选 **T4 GPU**
2. 左侧文件栏，把整个 `fine_tune` 文件夹拖进来
3. 从上到下逐个格子点 ▶ 运行

### 预估时间
- 基线测试：~15 分钟
- LoRA 微调：~40 分钟
- 微调后评估：~15 分钟

In [ ]:
# ── 1. 安装依赖 ──
!pip install -q transformers datasets accelerate peft trl bitsandbytes tensorboard

In [ ]:
# ── 2. 检查 GPU 和数据 ──
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

import os
for f in ['fine_tune/train_chatml.jsonl', 'fine_tune/validation_chatml.jsonl', 'fine_tune/test_chatml.jsonl']:
    if os.path.exists(f):
        n = sum(1 for _ in open(f))
        print(f'  ✓ {f} ({n} samples)')
    else:
        print(f'  ✗ {f} NOT FOUND — 请把 fine_tune 文件夹拖到左侧文件栏')

In [ ]:
# ── 3. 基线测试：基座模型零样本评估 ──
import json, re, time
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import output

MODEL = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Loading {MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

# Load test data
samples = []
with open('fine_tune/test_chatml.jsonl') as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))
print(f"Test samples: {len(samples)}")

def extract_fields(text):
    risk = re.search(r'风险等级:\s*(\S+)', text)
    enc = re.search(r'主要编码:\s*(\S+)', text)
    return {
        'risk_level': risk.group(1) if risk else '?',
        'encoding_primary': enc.group(1) if enc else '?',
    }

results = []
correct_risk = 0
t0 = time.time()

for idx, sample in enumerate(samples):
    msgs = sample['messages']
    gt = extract_fields(msgs[-1]['content'])
    
    prompt = tokenizer.apply_chat_template(
        [m for m in msgs if m['role'] in ('system','user')],
        tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.0,
                            do_sample=False, pad_token_id=tokenizer.pad_token_id)
    
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_fields(gen)
    
    risk_ok = pred['risk_level'] == gt['risk_level']
    enc_ok = pred['encoding_primary'] == gt['encoding_primary']
    if risk_ok: correct_risk += 1
    
    results.append({'idx':idx, 'gt_risk':gt['risk_level'], 'pred_risk':pred['risk_level'],
                    'risk_ok':risk_ok, 'gt_enc':gt['encoding_primary'], 'pred_enc':pred['encoding_primary'],
                    'enc_ok':enc_ok, 'generated':gen})
    
    if (idx+1) % 30 == 0:
        eta = (len(samples)-idx-1) * (time.time()-t0) / (idx+1)
        print(f"  [{idx+1}/{len(samples)}] risk_acc={correct_risk/(idx+1):.3f}  eta={eta:.0f}s")

elapsed = time.time() - t0
n = len(samples)
print(f"\nBaseline done in {elapsed:.0f}s ({n/elapsed:.1f} samples/s)")
print(f"Risk Accuracy: {correct_risk/n:.4f} ({correct_risk}/{n})")

# Save
!mkdir -p baseline_results
with open('baseline_results/baseline_details.jsonl','w') as f:
    for r in results: f.write(json.dumps(r,ensure_ascii=False)+'\n')

summary = {'model':MODEL, 'num_samples':n, 'risk_accuracy':correct_risk/n,
           'elapsed_seconds':round(elapsed,1)}
with open('baseline_results/baseline_summary.json','w') as f:
    json.dump(summary,f,indent=2,ensure_ascii=False)

print(f"\n基线准确率: {correct_risk/n:.2%}")
print(f"结果已保存到 baseline_results/")

In [ ]:
# ── 4. LoRA 微调 ──
import json, os
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
OUT_DIR = "./lora_output"

# Load data
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            if line.strip(): data.append(json.loads(line))
    return Dataset.from_list(data)

train_ds = load_jsonl('fine_tune/train_chatml.jsonl')
val_ds = load_jsonl('fine_tune/validation_chatml.jsonl')
print(f'train={len(train_ds)}  val={len(val_ds)}')

# Load model with 4-bit
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

# LoRA
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
                  bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

# Train
args = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=4, per_device_eval_batch_size=4,
    gradient_accumulation_steps=2, learning_rate=5e-5, num_train_epochs=3,
    warmup_ratio=0.1, logging_steps=20, save_steps=200, eval_steps=200,
    evaluation_strategy="steps", save_total_limit=2, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    fp16=True, remove_unused_columns=False, report_to="none",
)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                     tokenizer=tokenizer, max_seq_length=2048,
                     formatting_func=lambda x: x["messages"])

print("\nStarting LoRA training ...\n")
trainer.train()

# Save
os.makedirs(f'{OUT_DIR}/adapter', exist_ok=True)
trainer.model.save_pretrained(f'{OUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUT_DIR}/adapter')
print(f"\nLoRA weights saved to {OUT_DIR}/adapter")
print("Done.")

In [ ]:
# ── 5. 微调后评估：用训练好的 LoRA 再跑一次 test ──
from peft import PeftModel

MODEL = "Qwen/Qwen3-4B-Instruct-2507"

print("Loading base model + LoRA adapter ...")
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, 'lora_output/adapter')
model.eval()
print("Model loaded.")

samples = []
with open('fine_tune/test_chatml.jsonl') as f:
    for line in f:
        if line.strip(): samples.append(json.loads(line))

correct_risk = 0
t0 = time.time()

for idx, sample in enumerate(samples):
    msgs = sample['messages']
    gt = extract_fields(msgs[-1]['content'])
    prompt = tokenizer.apply_chat_template(
        [m for m in msgs if m['role'] in ('system','user')],
        tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    pred = extract_fields(gen)
    if pred['risk_level'] == gt['risk_level']: correct_risk += 1
    if (idx+1) % 30 == 0:
        print(f"  [{idx+1}/{len(samples)}] acc={correct_risk/(idx+1):.3f}")

n = len(samples)
tuned_acc = correct_risk / n

# Read baseline
with open('baseline_results/baseline_summary.json') as f:
    bl = json.load(f)

print(f"\n{'='*50}")
print(f"基线准确率 (zero-shot):    {bl['risk_accuracy']:.2%}")
print(f"微调后准确率 (LoRA):       {tuned_acc:.2%}")
print(f"提升:                       {(tuned_acc-bl['risk_accuracy'])*100:+.1f} 个百分点")
print(f"{'='*50}")

with open('baseline_results/tuned_summary.json','w') as f:
    json.dump({'baseline_acc':bl['risk_accuracy'], 'tuned_acc':tuned_acc, 'improvement':tuned_acc-bl['risk_accuracy']}, f, indent=2)

In [ ]:
# ── 6. 打包下载所有结果 ──
!zip -r results.zip baseline_results/ lora_output/adapter/
from google.colab import files
files.download('results.zip')